# Full training runs: our ViT, 0 vs 4 registers (Google Colab)

Trains ViT-Tiny on **all 100,000 Tiny-ImageNet images for 50 epochs**, twice:

| Run | Registers | Augmentation | Training images |
|---|---|---|---|
| `tiny_reg0` | 0 | basic (random crop + flip) | 100,000 |
| `tiny_reg4` | 4 | basic (random crop + flip) | 100,000 |

Then it makes a results table and graph.

**Time:** about 75 minutes per run, so roughly 2.5 hours in total.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Keep this tab open and your laptop awake while it runs.

**If Colab disconnects:** just do `Runtime → Run all` again. Progress is saved to your
Google Drive after every epoch, so training continues from where it stopped.

## 0. Setup: get the code

In [ ]:
import os

REPO_URL = "https://github.com/recommended131225435/vit-registers-reproduction.git"

if os.path.exists("../src") and not os.path.exists("src"):  # opened from inside notebooks/
    %cd ..
if not os.path.exists("src"):                                  # fresh Colab session
    !git clone -q {REPO_URL} repo
    %cd repo
else:
    !git pull -q                                               # make sure the code is up to date
!pip install -q -r requirements.txt

import torch
from IPython.display import Image, Markdown, display
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4!)")

## 1. Connect Google Drive
The trained models and logs are saved here, so nothing is lost if Colab disconnects. Click **Allow** when asked.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/vit-registers-runs"
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs will be saved in", RUN_DIR)

## 2. Train without registers (about 75 min)

In [ ]:
!python -m src.train --size tiny --registers 0 --epochs 50 --resume --out {RUN_DIR}/tiny_reg0

## 3. Train with 4 registers (about 75 min)

In [ ]:
!python -m src.train --size tiny --registers 4 --epochs 50 --resume --out {RUN_DIR}/tiny_reg4

## 4. Results table and graph

In [ ]:
!python -m scripts.summarise_training --runs {RUN_DIR}/tiny_reg0 {RUN_DIR}/tiny_reg4

display(Markdown(open("results/runs/full_training_summary.md").read().split("![")[0]))
display(Image("results/figures/full_training_runs.png"))

## 5. Download the results
The zip holds the logs, the graph and the updated README (not the model weights; those stay on your Google Drive).

In [ ]:
!zip -qr full_training_results.zip results/runs results/figures/full_training_runs.png README.md

from google.colab import files
files.download("full_training_results.zip")